### 0. Environment Setup
Install required libraries and download NLTK WordNet data for the synonym perturbation engine.

In [1]:
!pip install transformers datasets torch nltk pandas tqdm

import os
import re
import nltk
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.notebook import tqdm
from transformers import AutoModelForMaskedLM, AutoTokenizer

# Suppress Hugging Face warnings
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"

# Download WordNet for synonym substitution
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.corpus import wordnet as wn

### 1. Transformer (RoBERTa) Scorer
The core Masked Language Model evaluator. We use this to compute token-level pseudo-log-likelihood.

In [2]:
class MaskedLMPLLScorer:
    def __init__(self, model_name: str, device: torch.device):
        self.model_name = model_name
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
        self.model.eval()

    @torch.no_grad()
    def sentence_pll(self, sentence: str) -> float:
        enc = self.tokenizer(sentence, return_tensors="pt")
        input_ids = enc["input_ids"].to(self.device)
        attention_mask = enc["attention_mask"].to(self.device)
        seq_len = input_ids.size(1)
        pll = 0.0
        special_mask = self.tokenizer.get_special_tokens_mask(input_ids[0].tolist(), already_has_special_tokens=True)

        for pos in range(seq_len):
            if special_mask[pos] == 1: continue
            original_token_id = input_ids[0, pos].item()
            masked_ids = input_ids.clone()
            masked_ids[0, pos] = self.tokenizer.mask_token_id
            outputs = self.model(input_ids=masked_ids, attention_mask=attention_mask)
            logits = outputs.logits[0, pos]
            log_probs = torch.log_softmax(logits, dim=-1)
            pll += float(log_probs[original_token_id].item())
        return pll

    def compare_candidates(self, candidate1: str, candidate2: str) -> str:
        score1 = self.sentence_pll(candidate1)
        score2 = self.sentence_pll(candidate2)
        return "1" if score1 >= score2 else "2"

### 2. The Novelty: Linguistic Perturbation Engine
These functions introduce controlled modifications to the sentences to test if the model's reasoning is robust or fragile.

In [3]:
ADJ_POLARITY_MAP = {
    "big": "small", "small": "big", "large": "small", "tiny": "large",
    "old": "young", "young": "old", "strong": "weak", "weak": "strong",
    "heavy": "light", "light": "heavy", "wide": "narrow", "narrow": "wide",
    "long": "short", "short": "long", "high": "low", "low": "high",
    "rich": "poor", "poor": "rich",
}

def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def synonym_substitution(sentence: str) -> Optional[str]:
    tokens = re.findall(r"\w+|[^\w\s]", sentence)
    for i, tok in enumerate(tokens):
        if not re.match(r"^[A-Za-z]+$", tok): continue
        lower = tok.lower()
        synsets = wn.synsets(lower)
        candidates = []
        for syn in synsets:
            for lemma in syn.lemmas():
                name = lemma.name().replace("_", " ")
                if name.lower() != lower and " " not in name and name.isalpha():
                    candidates.append(name)
        if candidates:
            replacement = candidates[0].capitalize() if tok[0].isupper() else candidates[0]
            new_tokens = tokens[:]
            new_tokens[i] = replacement
            out = "".join([(t if re.match(r"[^\w\s]", t) else (" " + t)) for t in new_tokens]).strip()
            return normalize_space(out)
    return None

def adjective_polarity_change(sentence: str) -> Optional[str]:
    for src, tgt in ADJ_POLARITY_MAP.items():
        pattern = re.compile(rf"\b{re.escape(src)}\b", re.IGNORECASE)
        match = pattern.search(sentence)
        if match:
            matched = match.group(0)
            replacement = tgt.capitalize() if matched[0].isupper() else tgt
            return pattern.sub(replacement, sentence, count=1)
    return None

def apply_perturbations(sentence: str) -> Dict[str, str]:
    outputs = {}
    syn = synonym_substitution(sentence)
    if syn and syn != sentence: outputs["synonym"] = syn
    
    pol = adjective_polarity_change(sentence)
    if pol and pol != sentence: outputs["polarity"] = pol
    
    return outputs

### 3. Data Loading
Strictly extracts the first 300 sequential sentences from the WinoGrande validation set to match the baseline.

In [4]:
@dataclass
class Example:
    sentence: str
    option1: str
    option2: str
    answer: str
    example_id: str

def replace_blank(sentence: str, replacement: str) -> str:
    return normalize_space(sentence.replace("_", replacement, 1))

def load_winogrande_examples(split: str, max_items: int) -> List[Example]:
    ds = load_dataset("winogrande", "winogrande_xl", split=split)
    rows = list(ds)[:max_items]
    examples = []
    for idx, row in enumerate(rows):
        row_id = str(row.get("qID", row.get("id", idx)))
        examples.append(Example(
            sentence=row["sentence"], option1=row["option1"], 
            option2=row["option2"], answer=str(row["answer"]), example_id=row_id
        ))
    return examples

### 4. Main Experiment Execution (Upgraded to RoBERTa)
Evaluates the base accuracy, applies perturbations, recalculates accuracy, and flags flipped predictions.

In [5]:
def run_main_experiment():
    # UPGRADED MODEL: We are moving from BERT to RoBERTa-large for advanced semantic reasoning
    MODEL_NAME = "roberta-large"
    MAX_ITEMS = 300 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    scorer = MaskedLMPLLScorer(model_name=MODEL_NAME, device=device)
    examples = load_winogrande_examples("validation", MAX_ITEMS)
    print(f"Loaded the first {len(examples)} examples from WinoGrande.\n")
    
    results = []
    
    for ex in tqdm(examples, desc="Evaluating Perturbations"):
        # 1. Base Evaluation
        cand1 = replace_blank(ex.sentence, ex.option1)
        cand2 = replace_blank(ex.sentence, ex.option2)
        base_pred = scorer.compare_candidates(cand1, cand2)
        base_correct = (base_pred == ex.answer)
        
        # 2. Apply Perturbations
        perturbed_versions = apply_perturbations(ex.sentence)
        
        for p_type, p_sentence in perturbed_versions.items():
            try:
                p_cand1 = replace_blank(p_sentence, ex.option1)
                p_cand2 = replace_blank(p_sentence, ex.option2)
                p_pred = scorer.compare_candidates(p_cand1, p_cand2)
                p_correct = (p_pred == ex.answer)
                
                results.append({
                    "example_id": ex.example_id,
                    "perturbation_type": p_type,
                    "original_sentence": ex.sentence,
                    "perturbed_sentence": p_sentence,
                    "gold_answer": ex.answer,
                    "original_correct": base_correct,
                    "perturbed_correct": p_correct,
                    "changed_prediction": (base_pred != p_pred)
                })
            except Exception:
                continue 
                
    df_results = pd.DataFrame(results)
    
    print("\n--- MAIN EXPERIMENT RESULTS ---")
    summary = df_results.groupby("perturbation_type").agg(
        total_tested=("perturbation_type", "count"),
        original_accuracy=("original_correct", "mean"),
        perturbed_accuracy=("perturbed_correct", "mean"),
        changed_prediction_rate=("changed_prediction", "mean")
    ).reset_index()
    
    summary['original_accuracy'] = (summary['original_accuracy'] * 100).map('{:.2f}%'.format)
    summary['perturbed_accuracy'] = (summary['perturbed_accuracy'] * 100).map('{:.2f}%'.format)
    summary['changed_prediction_rate'] = (summary['changed_prediction_rate'] * 100).map('{:.2f}%'.format)
    
    print(summary.to_string(index=False))
    
    return df_results, summary

raw_df, summary_df = run_main_experiment()


Using device: cpu


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Loaded the first 300 examples from WinoGrande.



Evaluating Perturbations:   0%|          | 0/300 [00:00<?, ?it/s]


--- MAIN EXPERIMENT RESULTS ---
perturbation_type  total_tested original_accuracy perturbed_accuracy changed_prediction_rate
         polarity            36            58.33%             47.22%                  11.11%
          synonym           300            58.00%             56.00%                  12.00%


In [6]:
# Create a new cell and paste this code

# Filter for all cases where the perturbation caused the model to flip its prediction
flipped_df = raw_df[raw_df['changed_prediction'] == True]

# Separate them by type
polarity_flips = flipped_df[flipped_df['perturbation_type'] == 'polarity']
synonym_flips = flipped_df[flipped_df['perturbation_type'] == 'synonym']

print(f"=== POLARITY FLIPS ({len(polarity_flips)} instances) ===")
print("Context: Changing the adjective caused RoBERTa to change its answer.")
print("This suggests the model IS using semantic reasoning for these specific sentences.\n")

for _, row in polarity_flips.iterrows():
    print(f"Original  (Correct? {row['original_correct']}): {row['original_sentence']}")
    print(f"Perturbed (Correct? {row['perturbed_correct']}): {row['perturbed_sentence']}")
    print("-" * 80)

print(f"\n\n=== SYNONYM FLIPS ({len(synonym_flips)} instances) ===")
print("Context: Swapping a synonym caused RoBERTa to change its answer.")
print("This exposes the model's fragility and reliance on shallow lexical cues.\n")

# We use .head(10) to just show the first 10 so it doesn't flood your screen
for _, row in synonym_flips.head(10).iterrows():
    print(f"Original  (Correct? {row['original_correct']}): {row['original_sentence']}")
    print(f"Perturbed (Correct? {row['perturbed_correct']}): {row['perturbed_sentence']}")
    print("-" * 80)

=== POLARITY FLIPS (4 instances) ===
Context: Changing the adjective caused RoBERTa to change its answer.
This suggests the model IS using semantic reasoning for these specific sentences.

Original  (Correct? True): James panicked when his phone fell on the table thinking it will break but the _ is strong.
Perturbed (Correct? False): James panicked when his phone fell on the table thinking it will break but the _ is weak.
--------------------------------------------------------------------------------
Original  (Correct? True): I picked up some leaves to put in the books and dry, but they didn't fit because the _ were too small.
Perturbed (Correct? False): I picked up some leaves to put in the books and dry, but they didn't fit because the _ were too big.
--------------------------------------------------------------------------------
Original  (Correct? True): John never mentioned his canoe, but had a long conversation with Ron about the raft, because John rarely used the _ .
Perturbe